# 379. Design Phone Directory

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** linked-list, hash-table, design, queue
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-phone-directory/)

Design a phone directory that initially has `maxNumbers` empty slots. The
directory must hand out numbers, say whether a given slot is still empty, and
empty a slot again.

Implement the `PhoneDirectory` class:

- `PhoneDirectory(maxNumbers)` initializes the directory with `maxNumbers`
  available slots.
- `get()` provides a number that is **not assigned to anyone**. Returns `-1`
  if no number is available.
- `check(number)` returns `True` if the slot `number` is **available**, and
  `False` otherwise.
- `release(number)` recycles the slot `number`, making it available again.

---

### Example 1

```
Input:  ["PhoneDirectory", "get", "get", "check", "get", "check", "release", "check"]
        [[3],              [],    [],    [2],     [],    [2],     [2],       [2]]
Output: [null,             0,     1,     true,    2,     false,   null,      true]

PhoneDirectory phoneDirectory = new PhoneDirectory(3);
phoneDirectory.get();      // can return ANY available number, here assume 0
phoneDirectory.get();      // assume it returns 1
phoneDirectory.check(2);   // 2 has not been assigned yet, so return true
phoneDirectory.get();      // returns 2, the only number left
phoneDirectory.check(2);   // 2 is assigned now, so return false
phoneDirectory.release(2); // put 2 back in the pool
phoneDirectory.check(2);   // 2 is available again, return true
```

---

### Constraints

- `1 <= maxNumbers <= 10^4`
- `0 <= number < maxNumbers`
- At most `2 * 10^4` calls will be made to `get`, `check`, and `release`.

Read the `get()` line once more. "**A** number that is not assigned" - not the
smallest one, not a random one. *Any* of them. That one word is what lets all
three methods be `O(1)`, and it is also why the test cell below cannot be a
list of expected answers.


## Before you write anything

Your third **design** problem, and the smallest one yet: three methods, all
`O(1)` once you pick the right container. #155 came down to one well-placed
field per node, #355 to one clock. This one comes down to one well-chosen set -
so all the thinking is about *what you store*. Paper first.

**1.** The idea everyone has first: a list of booleans, `free[i] = True` while
slot `i` is unassigned. Which of the three methods is `O(1)` on it immediately,
and which one has to **scan**? With `maxNumbers = 10^4` and `2 * 10^4` calls,
multiply out the worst case for that scan. You will land in the hundreds of
millions - that number is why this problem exists.

**2.** Now name the container you actually want. Write the three methods as
three sentences that start with "take out one / test whether / put back", then
say which built-in Python type does all three in `O(1)` average. You have used
it since #242 to answer *is this thing in here?* - same type, two more jobs.

**3.** LeetCode tags this **linked-list**. Where would a linked list even go?
Draw it: a chain holding only the *free* numbers, `get` unhooks a node from one
end, `release` hooks one back on. Which end does each touch? Is `get` allowed
to take from the same end `release` just pushed onto - does the order you hand
numbers out change whether you are *correct*?

**4.** The trap, and it is the whole problem. Suppose your free pool is *only*
that chain (or a `deque`). Trace `maxNumbers = 2`:

```
get()      get()      release(0)      release(0)      get()      get()
```

Write what the pool holds after each step and what the last two `get`s return.
What just happened to the caller - two different people holding the same phone
number? Then: what one extra piece of state turns the second `release(0)` into
a **no-op**?

**5.** Push that answer one step. If you keep the chain *and* that extra state,
which of them does `check` read? Which does `get` read? So: do you still need
the chain at all? Answer in one line - what does route B buy that route A does
not?

**6.** `__init__` receives `maxNumbers` and every number starts free. What does
route A's constructor cost in time and space? Now suppose a directory is built
for `10^9` numbers and `get` is called five times. What would you want to store
instead? Sketch that state - you will build it in *After it passes*.

**7.** How do you test a method that is allowed to return **any** free number?
There is no expected output to write down - same wall as #382, for a completely
different reason. List what you *can* assert about every single `get()` answer.
There are three things, and one of them is about a number you handed the caller
several calls ago.


## Two routes - A is the answer, B is what the tag is pointing at

**A - one set of free numbers** *(write this first)*
`self.free = set(range(maxNumbers))`. `get` takes any element out, `check` is a
membership test, `release` puts one back. All three `O(1)` average, `O(n)`
space - and `release` is idempotent for free, because a set cannot hold the
same number twice, so question 4's trap cannot even be expressed. `set.pop()`
removes and returns *an arbitrary element*, which is exactly the freedom the
problem handed you. This is the version to submit.

**B - free list + an "is it free?" flag** *(the linked-list / queue answer)*
Keep the free numbers in a queue - your own linked list from #206, or a
`deque` - and beside it one boolean per slot. `get` pops the front and marks it
taken, `check` reads the boolean, `release` returns early if the boolean
already says free and otherwise flips it and pushes to the back. Same `O(1)`
everywhere, same `O(n)` space. Build it once: it is how you hand numbers out in
**arrival order** rather than arbitrary order, and the boolean array is the
guard route A got for nothing.

Look at what happened there. The flag list from question 1 was not wrong, it
was *incomplete*: it answers `check` in `O(1)` and cannot answer `get`. The
queue answers `get` in `O(1)` and cannot answer `check`. Route B is those two
side by side, each covering the other's blind spot - and the price of two
structures describing one fact is that every method must update **both** or
they drift apart. Route A's whole appeal is that there is only one fact.

Write A, run the tests, then rewrite the class as B and run the same tests -
the harness referees behaviour, so it does not care which one is underneath.


In [ ]:
import random
from collections import deque   # route B may want this; route A needs neither


In [ ]:
class PhoneDirectory:

    def __init__(self, maxNumbers: int):
    def get(self) -> int:
    def check(self, number: int) -> bool:
    def release(self, number: int) -> None:


### The test harness

`get()` may hand back any free number, so there is no output list to compare
against - question 7. Instead `replay` keeps its **own** set of free numbers, a
referee walking beside your class, and judges every answer against it:

- `get()` must return an `int` that is free *right now* - never one already
  handed out, never one outside `range(maxNumbers)`;
- `get()` must return `-1` exactly when the referee's set is empty;
- `check(n)` must agree with the referee, call for call;
- `release(n)` puts `n` back - and the referee is what catches you if a sloppy
  `release` lets the same number be handed out twice.

`stress` builds long random op sequences and referees them the same way; the
seed makes every run identical. On a failure you get the trace, with the
verdict as its last line. Run this cell; don't edit it.


In [ ]:
def replay(ops, args):
    """Replay a LeetCode-style (ops, args) sequence, refereeing every answer.

    Returns (ok, log). On failure the log ends with the line that broke.
    """
    pd, size, free, log = None, 0, set(), []

    for op, a in zip(ops, args):
        if op == "PhoneDirectory":
            size = a[0]
            pd, free = PhoneDirectory(size), set(range(size))
            log.append(f"PhoneDirectory({size})")

        elif op == "get":
            got = pd.get()
            log.append(f"get() -> {got!r}")
            if not free:
                if got != -1:
                    log.append("   !! nothing is free, so get() must return -1")
                    return False, log
            elif isinstance(got, bool) or not isinstance(got, int):
                log.append(f"   !! get() must return an int, got {type(got).__name__}")
                return False, log
            elif got not in free:
                why = "already assigned" if 0 <= got < size else f"outside range({size})"
                log.append(f"   !! get() returned {got}: {why}")
                return False, log
            else:
                free.discard(got)

        elif op == "check":
            n = a[0]
            got = pd.check(n)
            log.append(f"check({n}) -> {got!r}")
            if bool(got) != (n in free):
                log.append(f"   !! check({n}) must be {n in free}")
                return False, log

        elif op == "release":
            n = a[0]
            pd.release(n)
            free.add(n)
            log.append(f"release({n})")

    return True, log


def stress(size, calls, seed=0, mix=(0.45, 0.25)):
    """A long random op sequence on `size` slots, refereed by replay."""
    random.seed(seed)
    ops, args = ["PhoneDirectory"], [[size]]
    for _ in range(calls):
        r = random.random()
        if r < mix[0]:
            ops.append("get"), args.append([])
        elif r < mix[0] + mix[1]:
            ops.append("check"), args.append([random.randrange(size)])
        else:
            ops.append("release"), args.append([random.randrange(size)])
    return replay(ops, args)


def report(name, ok, log, tail=5):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")


In [ ]:
# tests
CASES = [
    ("the LeetCode example",
     ["PhoneDirectory", "get", "get", "check", "get", "check", "release", "check"],
     [[3],              [],    [],    [2],     [],    [2],     [2],       [2]]),

    ("one slot: hand it out, run dry, recycle it",
     ["PhoneDirectory", "check", "get", "check", "get", "release", "check", "get"],
     [[1],              [0],     [],    [0],     [],    [0],       [0],     []]),

    ("question 4's trap: release the same number twice",
     ["PhoneDirectory", "get", "get", "release", "release", "get", "get"],
     [[2],              [],    [],    [0],       [0],       [],    []]),

    ("release a number that was never assigned - the pool must not grow",
     ["PhoneDirectory", "release", "get", "get", "get", "get"],
     [[3],              [1],       [],    [],    [],    []]),

    ("check must not consume anything",
     ["PhoneDirectory", "check", "check", "check", "get", "get", "get"],
     [[2],              [0],     [0],     [1],     [],    [],    []]),

    ("drain it, give everything back, drain it again",
     ["PhoneDirectory", "get", "get", "get", "get", "get",
      "release", "release", "release", "release",
      "get", "get", "get", "get", "get"],
     [[4],              [],    [],    [],    [],    [],
      [0],       [1],       [2],       [3],
      [],    [],    [],    [],    []]),
]

for name, ops, args in CASES:
    report(name, *replay(ops, args))

# random sequences - the last one is the constraint ceiling from the statement.
# It should finish as fast as the others; if it visibly lags, something inside
# your class is scanning (question 1).
for size, calls, seed in [(1, 60, 1), (2, 200, 2), (8, 1000, 3),
                          (100, 4000, 4), (10000, 20000, 5)]:
    report(f"stress: {size} slots, {calls} random calls (seed {seed})",
           *stress(size, calls, seed))

# see it, do not just trust the pass/fail
print("\ntrace of the LeetCode example (your get() may pick other numbers):")
for line in replay(CASES[0][1], CASES[0][2])[1]:
    print("  " + line)


## After it passes

- **Build the `O(1)` constructor** (question 6). Route A pays `O(n)` time and
  `O(n)` memory before anyone has asked for a single number. Replace the set
  with two pieces of state: a counter `next` for how far you have ever gone,
  and a pool of *recycled* numbers. `get` prefers the pool, else hands out
  `next` and increments it, else returns `-1`. Now write `check(number)` - it
  has **two** cases, and dropping either one is the bug. `__init__` becomes
  `O(1)` and memory follows the numbers actually in play, not `maxNumbers`.
  Re-run the same tests.
- **FIFO or LIFO?** Route A hands out an arbitrary number, a `deque` pool hands
  out the *oldest* release, your linked-list stack hands out the *newest*. All
  three pass the harness. Which would a real phone company ship - and why is
  handing back the number you released a minute ago the wrong answer, even
  though it is the cheapest? (Think about who dials it next.)
- **`set.pop()` is arbitrary, not random.** It returns whatever the hash table
  hits first; do not read "arbitrary" as "uniform". #382 needed a real
  guarantee and had to earn it with a proof. What in *this* statement's wording
  lets you get away without one?
- **Two structures, one fact.** Route B stores availability twice, so every
  method must touch both. Where else did you keep a second structure in step
  with the first (#155's `min_stack`, #355's follow sets)? What broke, in each
  case, when one of the two updates was missing?
- **Complexities, per method:** write time and space for `__init__`, `get`,
  `check`, `release` - route A, route B, and the lazy version, side by side.
  One cell changes between them; the rest stay `O(1)`.
- Siblings: **#380 Insert Delete GetRandom O(1)** - this problem plus #382's
  uniform-random promise, which is precisely what a set cannot give you (the
  answer is an array plus an index map; do it next); #146 LRU Cache; #622
  Design Circular Queue (route B's pool on a fixed array); #705 Design HashSet
  (build the container route A leans on).
